# Heart Disease Prediction

In [20]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv('data/cleveland.csv')
df['disease'] = df['num'].apply(lambda x: True if x > 0 else False)

In [26]:
features = ['age', 'sex', 'trestbps', 'chol']

feat_df = df[features]

# Standardize features
for feature in features:
    feat_df[feature] = (feat_df[feature] - feat_df[feature].mean()) / feat_df[feature].std()

x_train, x_test, y_train, y_test = train_test_split(feat_df, df['disease'], test_size=0.2, random_state=63)

C:\Users\sjhar\AppData\Local\Temp\ipykernel_30776\2791777811.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  feat_df[feature] = (feat_df[feature] - feat_df[feature].mean()) / feat_df[feature].std()


In [35]:
from sklearn.model_selection import KFold
from sklearn.neighbors import NearestNeighbors

kfold = KFold(n_splits=10, shuffle=True, random_state=63)
precision_list = []
recall_list = []
f1_score_list = []
for i, (train_index, val_index) in enumerate(kfold.split(x_train, y_train)):
    print(f"Fold {i+1}:")
    x_df = x_train.iloc[train_index]
    x = x_df[features].values
    nn = NearestNeighbors(n_neighbors=5)
    fit = nn.fit(x)

    confusion_matrix = {
        "TP": 0,
        "TN": 0,
        "FP": 0,
        "FN": 0
    }

    for j in val_index:
        distances, indices = fit.kneighbors([x_train.iloc[j]])
        disease = 0
        healthy = 0
        for index in indices[0]:
            if y_train.iloc[index]:
                disease += 1
            else:
                healthy += 1
        if disease > healthy:
            if y_train.iloc[j]:
                confusion_matrix["TP"] += 1
            else:
                confusion_matrix["FP"] += 1
        else:
            if y_train.iloc[j]:
                confusion_matrix["FN"] += 1
            else:
                confusion_matrix["TN"] += 1

    print(f"Confusion Matrix: {confusion_matrix}")
    precision = confusion_matrix["TP"] / (confusion_matrix["TP"] + confusion_matrix["FP"]) if (confusion_matrix["TP"] + confusion_matrix["FP"]) > 0 else 0
    recall = confusion_matrix["TP"] / (confusion_matrix["TP"] + confusion_matrix["FN"]) if (confusion_matrix["TP"] + confusion_matrix["FN"]) > 0 else 0
    f1_score = 2 * (precision * recall) / (precision + recall)
    print(f"Precision: {precision:.4f}, Recall: {recall:.4f}, F1 Score: {f1_score:.4f}")

    precision_list.append(precision)
    recall_list.append(recall)
    f1_score_list.append(f1_score)

    print()

print(f"Average Precision: {sum(precision_list)/len(precision_list):.4f}")
print(f"Average Recall: {sum(recall_list)/len(recall_list):.4f}")
print(f"Average F1 Score: {sum(f1_score_list)/len(f1_score_list):.4f}")

Fold 1:
Confusion Matrix: {'TP': 3, 'TN': 11, 'FP': 7, 'FN': 4}
Precision: 0.3000, Recall: 0.4286, F1 Score: 0.3529

Fold 2:
Confusion Matrix: {'TP': 4, 'TN': 6, 'FP': 8, 'FN': 7}
Precision: 0.3333, Recall: 0.3636, F1 Score: 0.3478

Fold 3:
Confusion Matrix: {'TP': 3, 'TN': 8, 'FP': 5, 'FN': 8}
Precision: 0.3750, Recall: 0.2727, F1 Score: 0.3158

Fold 4:
Confusion Matrix: {'TP': 4, 'TN': 6, 'FP': 7, 'FN': 7}
Precision: 0.3636, Recall: 0.3636, F1 Score: 0.3636

Fold 5:
Confusion Matrix: {'TP': 4, 'TN': 9, 'FP': 4, 'FN': 7}
Precision: 0.5000, Recall: 0.3636, F1 Score: 0.4211

Fold 6:
Confusion Matrix: {'TP': 4, 'TN': 6, 'FP': 6, 'FN': 8}
Precision: 0.4000, Recall: 0.3333, F1 Score: 0.3636

Fold 7:
Confusion Matrix: {'TP': 2, 'TN': 7, 'FP': 8, 'FN': 7}
Precision: 0.2000, Recall: 0.2222, F1 Score: 0.2105

Fold 8:
Confusion Matrix: {'TP': 4, 'TN': 6, 'FP': 3, 'FN': 11}
Precision: 0.5714, Recall: 0.2667, F1 Score: 0.3636

Fold 9:
Confusion Matrix: {'TP': 5, 'TN': 5, 'FP': 7, 'FN': 7}
Precisi